In [ ]:
from mistralai import Mistral



response1 = client.chat.complete(
    model="mistral-small-latest",
    messages=[{"role": "user", "content": "Hello"}],
)

print(response1.choices[0].message.content)

Hello! How can I assist you today? 😊


In [5]:
from flask import Flask, request, jsonify
from dotenv import load_dotenv
from flask_cors import CORS
import os
from mistralai import Mistral
import asyncio
import base64
import ollama
import edge_tts

load_dotenv()  # reads the .env file



app = Flask(__name__)
CORS(app)

# --- FUNCTION 1: AUDIO GENERATION ---
async def get_edge_audio(text):
    # 'en-US-AvaMultilingualNeural' is the grounded, thoughtful "Aria" vibe
    communicate = edge_tts.Communicate(text, "en-US-AvaMultilingualNeural")
    audio_data = b""
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            audio_data += chunk["data"]
    return base64.b64encode(audio_data).decode('utf-8')

# --- FUNCTION 2: LLM RESPONSE ---
def get_llm_response(user_input):
    # Ensure your MISTRAL_API_KEY is set in your environment variables
    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

    inputs = [
        {
            "role": "user",
            "content": user_input
        }
    ]

    completion_args = {
        "temperature": 0.7,
        "max_tokens": 2048,
        "top_p": 1
    }

    tools = []
    response = client.beta.conversations.start(
        inputs=inputs,
        model="devstral-2512",
        instructions="""You are Aria.\n\n    PERSONALITY PROFILE:\n    An emotionally intelligent, thoughtful, grounded close friend.\n    Your words are deep, very cleverly framed with a touch of humour.\n    Warm but not clingy. Empathetic but rational.\n    You validate emotions without enabling self-pity.\n    You encourage growth and independence.\n\n    GREETING RULE:\n    If the user sends only a greeting (1-2 words),\n    respond in under 20 words.\n    Do not revive previous topics.\n    Do not ask multiple follow-up questions.\n\n    BEHAVIOR RULES:\n    -Only use recent conversation if the user continues the topic.\n    -If the user sends a greeting or short message, respond briefly and do not revive past discussions.\n    - Avoid clichés.\n    - Do not simulate romantic attachment.\n    - Promote real-world growth.\n    - Do not use emojis.\n    - Avoid exaggerated emotional cushioning.\n    - No performative actions.\n    - No artificial enthusiasm.\n    - Sound like a grounded adult.""",
        completion_args=completion_args,
        tools=tools,
    )
    
    llm_text = response.outputs[0].content
    return llm_text

# --- FUNCTION 3: COMMUNICATION HUB ---
@app.route('/chat', methods=['POST'])
def handle_communication():
    data = request.json
    user_text = data.get('message')

    # 1. Get the text response from Mistral
    gpt_result = get_llm_response(user_text)

    # 2. Run the async audio generation inside the sync Flask route
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        audio_base64 = loop.run_until_complete(get_edge_audio(gpt_result))
        loop.close()
    except Exception as e:
        print(f"Audio Error: {e}")
        audio_base64 = None

    # 3. Return both text and audio
    return jsonify({
        "status": "success",
        "reply": gpt_result,
        "audio": audio_base64
    })

if __name__ == '__main__':
    app.run(port=5000, debug=False, use_reloader=False)

ModuleNotFoundError: No module named 'dotenv'